In [1]:
from pathlib import Path
import pandas as pd

# Project paths
PROJECT_ROOT = Path.cwd().parent
RAW_DATA = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA = PROJECT_ROOT / "data" / "processed"

# Create processed folder if it doesn't exist
PROCESSED_DATA.mkdir(parents=True, exist_ok=True)

# Load all datasets
datasets = {
    "customers": pd.read_csv(RAW_DATA / "olist_customers_dataset.csv"),
    "geolocation": pd.read_csv(RAW_DATA / "olist_geolocation_dataset.csv"),
    "order_items": pd.read_csv(RAW_DATA / "olist_order_items_dataset.csv"),
    "order_payments": pd.read_csv(RAW_DATA / "olist_order_payments_dataset.csv"),
    "order_reviews": pd.read_csv(RAW_DATA / "olist_order_reviews_dataset.csv"),
    "orders": pd.read_csv(RAW_DATA / "olist_orders_dataset.csv"),
    "products": pd.read_csv(RAW_DATA / "olist_products_dataset.csv"),
    "sellers": pd.read_csv(RAW_DATA / "olist_sellers_dataset.csv"),
    "category_translation": pd.read_csv(
        RAW_DATA / "product_category_name_translation.csv"
    )
}

print("All datasets loaded successfully!")

All datasets loaded successfully!


In [2]:
for name, df in datasets.items():
    print(f"{name:20} {df.shape}")

customers            (99447, 5)
geolocation          (1000163, 5)
order_items          (112650, 7)
order_payments       (103886, 5)
order_reviews        (99224, 7)
orders               (99444, 8)
products             (32951, 9)
sellers              (3095, 4)
category_translation (71, 2)


In [3]:
# Check duplicate rows in each dataset

for name, df in datasets.items():
    duplicate_count = df.duplicated().sum()
    print(f"{name:20} Duplicate rows: {duplicate_count:,}")

customers            Duplicate rows: 2
geolocation          Duplicate rows: 261,831
order_items          Duplicate rows: 0
order_payments       Duplicate rows: 0
order_reviews        Duplicate rows: 0
orders               Duplicate rows: 1
products             Duplicate rows: 0
sellers              Duplicate rows: 0
category_translation Duplicate rows: 0


In [4]:
# Remove exact duplicate rows

for name in datasets:
    before = len(datasets[name])

    datasets[name] = datasets[name].drop_duplicates().reset_index(drop=True)

    after = len(datasets[name])
    removed = before - after

    print(f"{name:20} Removed: {removed:,} | New rows: {after:,}")

customers            Removed: 2 | New rows: 99,445
geolocation          Removed: 261,831 | New rows: 738,332
order_items          Removed: 0 | New rows: 112,650
order_payments       Removed: 0 | New rows: 103,886
order_reviews        Removed: 0 | New rows: 99,224
orders               Removed: 1 | New rows: 99,443
products             Removed: 0 | New rows: 32,951
sellers              Removed: 0 | New rows: 3,095
category_translation Removed: 0 | New rows: 71


In [5]:
# Verify duplicate removal

for name, df in datasets.items():
    print(f"{name:20} Duplicate rows remaining: {df.duplicated().sum():,}")

customers            Duplicate rows remaining: 0
geolocation          Duplicate rows remaining: 0
order_items          Duplicate rows remaining: 0
order_payments       Duplicate rows remaining: 0
order_reviews        Duplicate rows remaining: 0
orders               Duplicate rows remaining: 0
products             Duplicate rows remaining: 0
sellers              Duplicate rows remaining: 0
category_translation Duplicate rows remaining: 0


In [6]:
# Check missing values in all datasets

for name, df in datasets.items():
    missing = df.isnull().sum()
    missing = missing[missing > 0]

    print(f"\n{name}")
    
    if missing.empty:
        print("No missing values")
    else:
        for column, count in missing.items():
            percentage = (count / len(df)) * 100
            print(f"{column:40} {count:>8,} ({percentage:.2f}%)")


customers
customer_id                                     4 (0.00%)
customer_zip_code_prefix                        4 (0.00%)
customer_city                                   4 (0.00%)
customer_state                                  4 (0.00%)

geolocation
No missing values

order_items
No missing values

order_payments
No missing values

order_reviews
review_comment_title                       87,656 (88.34%)
review_comment_message                     58,247 (58.70%)

orders
order_id                                        2 (0.00%)
order_status                                    2 (0.00%)
order_purchase_timestamp                        2 (0.00%)
order_approved_at                             162 (0.16%)
order_delivered_carrier_date                1,785 (1.79%)
order_delivered_customer_date               2,967 (2.98%)
order_estimated_delivery_date                   2 (0.00%)

products
product_category_name                         610 (1.85%)
product_name_lenght                           

In [7]:
customers = datasets["customers"]

customers[customers["customer_id"].isna()]

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
99441,NaN,99441,NaN,NaN,NaN
99442,NaN,0,NaN,NaN,NaN
99443,NaN,3345,NaN,NaN,NaN
99444,NaN,1,NaN,NaN,NaN


In [8]:
customers[customers["customer_id"].isna()].T

,99441,99442,99443,99444
customer_id,NaN,NaN,NaN,NaN
customer_unique_id,99441,0,3345,1
customer_zip_code_prefix,NaN,NaN,NaN,NaN
customer_city,NaN,NaN,NaN,NaN
customer_state,NaN,NaN,NaN,NaN


In [9]:
# Check orders with missing customer_id

orders = datasets["orders"]

orders[orders["customer_id"].isna()]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date


In [10]:
# Count orders with missing customer_id

orders["customer_id"].isna().sum()

np.int64(0)

In [11]:
# Remove customers with missing customer_id

customers = datasets["customers"]

before = len(customers)

customers = customers.dropna(subset=["customer_id"]).reset_index(drop=True)

datasets["customers"] = customers

after = len(customers)

print(f"Customers before: {before:,}")
print(f"Customers after:  {after:,}")
print(f"Rows removed:     {before - after:,}")

Customers before: 99,445
Customers after:  99,441
Rows removed:     4


In [12]:
print("Missing customer_id:", datasets["customers"]["customer_id"].isna().sum())

Missing customer_id: 0


In [13]:
orders = datasets["orders"]

orders[orders["order_id"].isna()].T

,99441,99442
order_id,NaN,NaN
customer_id,99441,0
order_status,NaN,NaN
order_purchase_timestamp,NaN,NaN
order_approved_at,NaN,NaN
order_delivered_carrier_date,NaN,NaN
order_delivered_customer_date,NaN,NaN
order_estimated_delivery_date,NaN,NaN


In [14]:
orders[orders["order_id"].isna()].shape

(2, 8)

In [15]:
orders[orders["order_id"].isna()].T

,99441,99442
order_id,NaN,NaN
customer_id,99441,0
order_status,NaN,NaN
order_purchase_timestamp,NaN,NaN
order_approved_at,NaN,NaN
order_delivered_carrier_date,NaN,NaN
order_delivered_customer_date,NaN,NaN
order_estimated_delivery_date,NaN,NaN


In [16]:
# Remove orders with missing order_id

orders = datasets["orders"]

before = len(orders)

orders = orders.dropna(subset=["order_id"]).reset_index(drop=True)

datasets["orders"] = orders

after = len(orders)

print(f"Orders before: {before:,}")
print(f"Orders after:  {after:,}")
print(f"Rows removed:  {before - after:,}")

Orders before: 99,443
Orders after:  99,441
Rows removed:  2


In [17]:
print("Missing order_id:", datasets["orders"]["order_id"].isna().sum())

Missing order_id: 0


In [18]:
# Check order status distribution

orders = datasets["orders"]

print(orders["order_status"].value_counts(dropna=False))

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64


In [20]:
# Compare missing delivery dates with order status

delivery_check = orders.groupby("order_status").agg(
    total_orders=("order_id", "count"),
    missing_approved=("order_approved_at", lambda x: x.isna().sum()),
    missing_carrier_date=("order_delivered_carrier_date", lambda x: x.isna().sum()),
    missing_customer_date=("order_delivered_customer_date", lambda x: x.isna().sum())
)

delivery_check

,total_orders,missing_approved,missing_carrier_date,missing_customer_date
order_status,,,,
approved,2,0,2,2
canceled,625,141,550,619
created,5,5,5,5
delivered,96478,14,2,8
invoiced,314,0,314,314
processing,301,0,301,301
shipped,1107,0,0,1107
unavailable,609,0,609,609


In [21]:
# Inspect delivered orders with missing delivery timestamps

delivered_issues = orders[
    (orders["order_status"] == "delivered") &
    (
        orders["order_approved_at"].isna() |
        orders["order_delivered_carrier_date"].isna() |
        orders["order_delivered_customer_date"].isna()
    )
]

print(f"Delivered orders with missing dates: {len(delivered_issues)}")

delivered_issues

Delivered orders with missing dates: 23


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,ec05a6d8558c6455f0cbbd8a420ad34f,delivered,11/28/17 17:44,11/28/17 17:56,11/30/17 18:12,NaN,12-18-17
5323,e04abd8149ef81b95221e88f6ed9ab6a,2127dc6603ac33544953ef05ec155771,delivered,2/18/17 14:39,NaN,2/23/17 12:04,3/1/17 13:25,03-17-17
16567,8a9adc69528e1001fc68dd0aaebbb54a,4c1ccc74e00993733742a3c786dc3c1f,delivered,2/18/17 12:45,NaN,2/23/17 9:01,3/2/17 10:05,03-21-17
19031,7013bcfc1c97fe719a7b5e05e61c12db,2941af76d38100e0f8740a374f1a5dc3,delivered,2/18/17 13:29,NaN,2/22/17 16:25,3/1/17 8:07,03-17-17
20618,f5dd62b788049ad9fc0526e3ad11a097,5e89028e024b381dc84a13a3570decb4,delivered,6/20/18 6:58,6/20/18 7:19,6/25/18 8:05,NaN,07-16-18
22663,5cf925b116421afa85ee25e99b4c34fb,29c35fc91fc13fb5073c8f30505d860d,delivered,2/18/17 16:48,NaN,2/22/17 11:23,3/9/17 7:28,03-31-17
23156,12a95a3c06dbaec84bcfb0e2da5d228a,1e101e0daffaddce8159d25a8e53f2b2,delivered,2/17/17 13:05,NaN,2/22/17 11:23,3/2/17 11:09,03-20-17
26800,c1d4211b3dae76144deccd6c74144a88,684cb238dc5b5d6366244e0e0776b450,delivered,1/19/17 12:48,NaN,1/25/17 14:56,1/30/17 18:16,03-01-17
38290,d69e5d356402adc8cf17e08b5033acfb,68d081753ad4fe22fc4d410a9eb1ca01,delivered,2/19/17 1:28,NaN,2/23/17 3:11,3/2/17 3:41,03-27-17
39334,d77031d6a3c8a52f019764e68f211c69,0bf35cac6cc7327065da879e2d90fae8,delivered,2/18/17 11:04,NaN,2/23/17 7:23,3/2/17 16:15,03-22-17


In [22]:
# Display the problematic delivered orders clearly

delivered_issues.T

,3002,5323,16567,19031,20618,22663,23156,26800,38290,39334,...,63052,67697,72407,73222,79263,82868,84999,92643,97647,98038
order_id,2d1e2d5bf4dc7227b3bfebb81328c15f,e04abd8149ef81b95221e88f6ed9ab6a,8a9adc69528e1001fc68dd0aaebbb54a,7013bcfc1c97fe719a7b5e05e61c12db,f5dd62b788049ad9fc0526e3ad11a097,5cf925b116421afa85ee25e99b4c34fb,12a95a3c06dbaec84bcfb0e2da5d228a,c1d4211b3dae76144deccd6c74144a88,d69e5d356402adc8cf17e08b5033acfb,d77031d6a3c8a52f019764e68f211c69,...,51eb2eebd5d76a24625b31c33dd41449,88083e8f64d95b932164187484d90212,3c0b8706b065f9919d0505d3b3343881,2aa91108853cecb43c84a5dc5b277475,e69f75a717d64fc5ecdfae42b2e8e086,0d3268bad9b086af767785e3f0fc0133,2babbb4b15e6d2dfe95e2de765c97bce,2d858f451373b04fb5c984a1cc2defaf,ab7c89dc1bf4a1ead9d6ec1ec8968a84,20edc82cf5400ce95e1afacc25798b31
customer_id,ec05a6d8558c6455f0cbbd8a420ad34f,2127dc6603ac33544953ef05ec155771,4c1ccc74e00993733742a3c786dc3c1f,2941af76d38100e0f8740a374f1a5dc3,5e89028e024b381dc84a13a3570decb4,29c35fc91fc13fb5073c8f30505d860d,1e101e0daffaddce8159d25a8e53f2b2,684cb238dc5b5d6366244e0e0776b450,68d081753ad4fe22fc4d410a9eb1ca01,0bf35cac6cc7327065da879e2d90fae8,...,07a2a7e0f63fd8cb757ed77d4245623c,f67cd1a215aae2a1074638bbd35a223a,d85919cb3c0529589c6fa617f5f43281,afeb16c7f46396c0ed54acb45ccaaa40,cfda40ca8dd0a5d486a9635b611b398a,4f1d63d35fb7c8999853b2699f5c7649,74bebaf46603f9340e3b50c6b086f992,e08caf668d499a6d643dafd7c5cc498a,dd1b84a7286eb4524d52af4256c0ba24,28c37425f1127d887d7337f284080a0f
order_status,delivered,delivered,delivered,delivered,delivered,delivered,delivered,delivered,delivered,delivered,...,delivered,delivered,delivered,delivered,delivered,delivered,delivered,delivered,delivered,delivered
order_purchase_timestamp,11/28/17 17:44,2/18/17 14:39,2/18/17 12:45,2/18/17 13:29,6/20/18 6:58,2/18/17 16:48,2/17/17 13:05,1/19/17 12:48,2/19/17 1:28,2/18/17 11:04,...,2/18/17 15:52,2/18/17 22:49,2/17/17 15:53,9/29/17 8:52,7/1/18 22:05,7/1/18 21:14,2/18/17 17:15,5/25/17 23:22,6/8/18 12:09,6/27/18 16:09
order_approved_at,11/28/17 17:56,NaN,NaN,NaN,6/20/18 7:19,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,9/29/17 9:07,7/1/18 22:15,7/1/18 21:29,NaN,5/25/17 23:30,6/8/18 12:36,6/27/18 16:29
order_delivered_carrier_date,11/30/17 18:12,2/23/17 12:04,2/23/17 9:01,2/22/17 16:25,6/25/18 8:05,2/22/17 11:23,2/22/17 11:23,1/25/17 14:56,2/23/17 3:11,2/23/17 7:23,...,2/23/17 3:09,2/22/17 11:31,2/22/17 11:31,NaN,7/3/18 13:57,7/3/18 9:27,2/22/17 11:23,NaN,6/12/18 14:10,7/3/18 19:25
order_delivered_customer_date,NaN,3/1/17 13:25,3/2/17 10:05,3/1/17 8:07,NaN,3/9/17 7:28,3/2/17 11:09,1/30/17 18:16,3/2/17 3:41,3/2/17 16:15,...,3/7/17 13:57,3/2/17 12:06,3/3/17 11:47,11/20/17 19:44,NaN,NaN,3/3/17 18:43,NaN,NaN,NaN
order_estimated_delivery_date,12-18-17,03-17-17,03-21-17,03-17-17,07-16-18,03-31-17,03-20-17,03-01-17,03-27-17,03-22-17,...,03-29-17,03-21-17,03-23-17,11-14-17,07-30-18,07-24-18,03-31-17,06-23-17,06-26-18,07-19-18


In [23]:
# Investigate products with missing category information

products = datasets["products"]

missing_category = products[products["product_category_name"].isna()]

print(f"Products with missing category: {len(missing_category)}")

missing_category.head()

Products with missing category: 610


,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
105,a41e356c76fab66334f36de622ecbd3a,NaN,NaN,NaN,NaN,650.0,17.0,14.0,12.0
128,d8dee61c2034d6d075997acef1870e9b,NaN,NaN,NaN,NaN,300.0,16.0,7.0,20.0
145,56139431d72cd51f19eb9f7dae4d1617,NaN,NaN,NaN,NaN,200.0,20.0,20.0,20.0
154,46b48281eb6d663ced748f324108c733,NaN,NaN,NaN,NaN,18500.0,41.0,30.0,41.0
197,5fb61f482620cb672f5e586bb132eae9,NaN,NaN,NaN,NaN,300.0,35.0,7.0,12.0


In [24]:
# Check whether products with missing category are used in orders

order_items = datasets["order_items"]

missing_category_ids = set(
    products.loc[products["product_category_name"].isna(), "product_id"]
)

used_missing_category = order_items[
    order_items["product_id"].isin(missing_category_ids)
]

print(f"Products with missing category: {len(missing_category_ids):,}")
print(f"Order items using these products: {len(used_missing_category):,}")
print(f"Unique missing-category products used: "
      f"{used_missing_category['product_id'].nunique():,}")

Products with missing category: 610
Order items using these products: 1,603
Unique missing-category products used: 610


In [25]:
# Investigate products with missing physical attributes

physical_columns = [
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

physical_issues = products[
    products[physical_columns].isna().any(axis=1)
]

print(f"Products with missing physical attributes: {len(physical_issues)}")

physical_issues.T

Products with missing physical attributes: 2


,8578,18851
product_id,09ff539a621711667c43eba6a3bd8466,5eb564652db742ff8f28759cd8d2652a
product_category_name,bebes,NaN
product_name_lenght,60.0,NaN
product_description_lenght,865.0,NaN
product_photos_qty,3.0,NaN
product_weight_g,NaN,NaN
product_length_cm,NaN,NaN
product_height_cm,NaN,NaN
product_width_cm,NaN,NaN


In [26]:
# Check current data types

for name, df in datasets.items():
    print(f"\n{name}")
    print(df.dtypes)


customers
customer_id                 str
customer_unique_id          str
customer_zip_code_prefix    str
customer_city               str
customer_state              str
dtype: object

geolocation
geolocation_zip_code_prefix      int64
geolocation_lat                float64
geolocation_lng                float64
geolocation_city                   str
geolocation_state                  str
dtype: object

order_items
order_id                   str
order_item_id            int64
product_id                 str
seller_id                  str
shipping_limit_date        str
price                  float64
freight_value          float64
dtype: object

order_payments
order_id                    str
payment_sequential        int64
payment_type                str
payment_installments      int64
payment_value           float64
dtype: object

order_reviews
review_id                    str
order_id                     str
review_score               int64
review_comment_title         str
review_comme

In [27]:
# Check orders data types

orders = datasets["orders"]

print(orders.dtypes)

order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object


In [28]:
# Show the actual date format

print(orders[
    [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
].head())

  order_purchase_timestamp order_approved_at order_delivered_carrier_date  \
0            10/2/17 10:56     10/2/17 11:07                10/4/17 19:54   
1            7/24/18 20:41      7/26/18 3:24                7/26/18 14:30   
2              8/8/18 8:38       8/8/18 8:55                 8/8/18 13:50   
3           11/18/17 19:28    11/18/17 19:45               11/22/17 13:39   
4            2/13/18 21:18     2/13/18 22:20                2/14/18 19:46   

  order_delivered_customer_date order_estimated_delivery_date  
0                10/10/17 21:25                      10-18-17  
1                  8/7/18 15:27                      08-13-18  
2                 8/17/18 18:06                      09-04-18  
3                  12/2/17 0:28                      12-15-17  
4                 2/16/18 18:17                      02-26-18  


In [29]:
# Convert order date columns to datetime

orders = datasets["orders"]

date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for column in date_columns:
    orders[column] = pd.to_datetime(
        orders[column],
        errors="coerce"
    )

datasets["orders"] = orders

print(orders[date_columns].dtypes)

C:\Users\athar\AppData\Local\Temp\ipykernel_9640\1254150811.py:14: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  orders[column] = pd.to_datetime(
C:\Users\athar\AppData\Local\Temp\ipykernel_9640\1254150811.py:14: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  orders[column] = pd.to_datetime(
C:\Users\athar\AppData\Local\Temp\ipykernel_9640\1254150811.py:14: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  orders[column] = pd.to_datetime(
C:\Users\athar\AppData\Local\Temp\ipykernel_9640\1254150811.py:14: UserWarning: Could not infer format, so each element will be parsed individ

order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object


C:\Users\athar\AppData\Local\Temp\ipykernel_9640\1254150811.py:14: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  orders[column] = pd.to_datetime(


In [30]:
# Check missing values after date conversion

for column in date_columns:
    print(f"{column:40} {orders[column].isna().sum():,}")

order_purchase_timestamp                 0
order_approved_at                        160
order_delivered_carrier_date             1,783
order_delivered_customer_date            2,965
order_estimated_delivery_date            0


In [31]:
# Convert order_items shipping limit date to datetime

order_items = datasets["order_items"]

order_items["shipping_limit_date"] = pd.to_datetime(
    order_items["shipping_limit_date"],
    errors="coerce"
)

datasets["order_items"] = order_items

print(order_items["shipping_limit_date"].dtype)
print("Missing values:", order_items["shipping_limit_date"].isna().sum())

C:\Users\athar\AppData\Local\Temp\ipykernel_9640\3899979596.py:5: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  order_items["shipping_limit_date"] = pd.to_datetime(


datetime64[us]
Missing values: 0


In [32]:
order_items["shipping_limit_date"] = pd.to_datetime(
    order_items["shipping_limit_date"],
    format="%m/%d/%y %H:%M",
    errors="coerce"
)

In [33]:
print(order_items["shipping_limit_date"].dtype)
print("Missing values:", order_items["shipping_limit_date"].isna().sum())

datetime64[us]
Missing values: 0


In [34]:
order_payments = datasets["order_payments"]

print(order_payments.dtypes)

order_id                    str
payment_sequential        int64
payment_type                str
payment_installments      int64
payment_value           float64
dtype: object


In [35]:
order_reviews = datasets["order_reviews"]

print(order_reviews.dtypes)

review_id                    str
order_id                     str
review_score               int64
review_comment_title         str
review_comment_message       str
review_creation_date         str
review_answer_timestamp      str
dtype: object


In [36]:
# Convert review date columns to datetime

order_reviews = datasets["order_reviews"]

review_date_columns = [
    "review_creation_date",
    "review_answer_timestamp"
]

for column in review_date_columns:
    order_reviews[column] = pd.to_datetime(
        order_reviews[column],
        errors="coerce"
    )

datasets["order_reviews"] = order_reviews

print(order_reviews[review_date_columns].dtypes)

C:\Users\athar\AppData\Local\Temp\ipykernel_9640\3773104437.py:11: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  order_reviews[column] = pd.to_datetime(
C:\Users\athar\AppData\Local\Temp\ipykernel_9640\3773104437.py:11: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  order_reviews[column] = pd.to_datetime(


review_creation_date       datetime64[us]
review_answer_timestamp    datetime64[us]
dtype: object


In [37]:
for column in review_date_columns:
    print(f"{column}: {order_reviews[column].isna().sum():,} missing")

review_creation_date: 0 missing
review_answer_timestamp: 0 missing


In [38]:
products = datasets["products"]

print(products.dtypes)

product_id                        str
product_category_name             str
product_name_lenght           float64
product_description_lenght    float64
product_photos_qty            float64
product_weight_g              float64
product_length_cm             float64
product_height_cm             float64
product_width_cm              float64
dtype: object


In [39]:
print(datasets["customers"].dtypes)
print("\n--- SELLERS ---")
print(datasets["sellers"].dtypes)

customer_id                 str
customer_unique_id          str
customer_zip_code_prefix    str
customer_city               str
customer_state              str
dtype: object

--- SELLERS ---
seller_id                   str
seller_zip_code_prefix    int64
seller_city                 str
seller_state                str
dtype: object


In [40]:
# Referential integrity checks

customers = datasets["customers"]
orders = datasets["orders"]
order_items = datasets["order_items"]
products = datasets["products"]
sellers = datasets["sellers"]
order_payments = datasets["order_payments"]
order_reviews = datasets["order_reviews"]

print("Orders → Customers:",
      (~orders["customer_id"].isin(customers["customer_id"])).sum())

print("Order Items → Orders:",
      (~order_items["order_id"].isin(orders["order_id"])).sum())

print("Order Items → Products:",
      (~order_items["product_id"].isin(products["product_id"])).sum())

print("Order Items → Sellers:",
      (~order_items["seller_id"].isin(sellers["seller_id"])).sum())

print("Payments → Orders:",
      (~order_payments["order_id"].isin(orders["order_id"])).sum())

print("Reviews → Orders:",
      (~order_reviews["order_id"].isin(orders["order_id"])).sum())

Orders → Customers: 4
Order Items → Orders: 3
Order Items → Products: 0
Order Items → Sellers: 6
Payments → Orders: 5
Reviews → Orders: 8


In [41]:
# Investigate orders with missing customer references

orphan_customer_orders = orders[
    ~orders["customer_id"].isin(customers["customer_id"])
]

print(f"Orphan orders: {len(orphan_customer_orders)}")

orphan_customer_orders.T

Orphan orders: 4


,50690,55487,59139,90122
order_id,f61d0a25c102875f482d4ead106e18b1,21533e635256523fb56f1ce0d97439b9,b9a505c23bb00c07421e4ef2c9381e5b,d985f28d9d0fc8b82d89fe5fc272fc64
customer_id,"""56cabab28ceaf3ebe6980ce08fe843ba""",89bd3cf6475e5036e8e417aa3851e548,"""6cd80ca371f4be9a7bd93aefcf6dfcb7""","""42b60323285c89fc7c84d6cc24f515be"""
order_status,delivered,shipped,delivered,delivered
order_purchase_timestamp,2018-04-29 22:23:00,2017-11-07 21:45:00,2018-07-16 20:54:00,2018-01-31 06:57:00
order_approved_at,2018-04-29 22:35:00,2017-11-07 21:55:00,2018-07-16 21:10:00,2018-01-31 14:04:00
order_delivered_carrier_date,2018-04-30 16:00:00,2017-11-09 15:56:00,2018-07-17 10:34:00,2018-02-05 22:35:00
order_delivered_customer_date,2018-05-07 19:35:00,NaT,2018-07-26 18:21:00,2018-02-22 17:59:00
order_estimated_delivery_date,2018-05-17 00:00:00,2017-11-30 00:00:00,2018-08-06 00:00:00,2018-02-26 00:00:00


In [42]:
print(orphan_customer_orders[["order_id", "customer_id", "order_status"]])

                               order_id                         customer_id  \
50690  f61d0a25c102875f482d4ead106e18b1  "56cabab28ceaf3ebe6980ce08fe843ba"   
55487  21533e635256523fb56f1ce0d97439b9    89bd3cf6475e5036e8e417aa3851e548   
59139  b9a505c23bb00c07421e4ef2c9381e5b  "6cd80ca371f4be9a7bd93aefcf6dfcb7"   
90122  d985f28d9d0fc8b82d89fe5fc272fc64  "42b60323285c89fc7c84d6cc24f515be"   

      order_status  
50690    delivered  
55487      shipped  
59139    delivered  
90122    delivered  


In [43]:
# Check the customer IDs after removing quotation marks

orphan_ids = orphan_customer_orders["customer_id"].dropna().unique()

for customer_id in orphan_ids:
    cleaned_id = customer_id.strip('"')
    
    print("Original :", repr(customer_id))
    print("Cleaned  :", repr(cleaned_id))
    print("Found in customers:", cleaned_id in customers["customer_id"].values)
    print()

Original : '"56cabab28ceaf3ebe6980ce08fe843ba"'
Cleaned  : '56cabab28ceaf3ebe6980ce08fe843ba'
Found in customers: True

Original : '89bd3cf6475e5036e8e417aa3851e548'
Cleaned  : '89bd3cf6475e5036e8e417aa3851e548'
Found in customers: False

Original : '"6cd80ca371f4be9a7bd93aefcf6dfcb7"'
Cleaned  : '6cd80ca371f4be9a7bd93aefcf6dfcb7'
Found in customers: True

Original : '"42b60323285c89fc7c84d6cc24f515be"'
Cleaned  : '42b60323285c89fc7c84d6cc24f515be'
Found in customers: True



In [44]:
# Check quotation marks in customer IDs

print("Customers with quotes:",
      customers["customer_id"].astype(str).str.contains('"').sum())

print("Orders with quotes:",
      orders["customer_id"].astype(str).str.contains('"').sum())

Customers with quotes: 1
Orders with quotes: 3


In [45]:
# Remove unnecessary quotation marks from customer IDs

customers["customer_id"] = customers["customer_id"].str.strip('"')
orders["customer_id"] = orders["customer_id"].str.strip('"')

datasets["customers"] = customers
datasets["orders"] = orders

print("Customer IDs standardized successfully.")

Customer IDs standardized successfully.


In [46]:
# Re-check Orders → Customers relationship

orphan_customer_orders = orders[
    ~orders["customer_id"].isin(customers["customer_id"])
]

print(f"Orders with missing customer reference: {len(orphan_customer_orders)}")

Orders with missing customer reference: 0


In [47]:
# Investigate order items with missing order references

orphan_order_items = order_items[
    ~order_items["order_id"].isin(orders["order_id"])
]

print(f"Orphan order items: {len(orphan_order_items)}")

orphan_order_items.T

Orphan order items: 3


,11723,48065,48066
order_id,1a9b5dd397fc3ec9e54ff4b9a738a37e,6d34c087b843ae0fdb296f15d948e45f,6d34c087b843ae0fdb296f15d948e45f
order_item_id,1,1,2
product_id,88fb259945a739dec6cdd001e2cbad55,4b5d91666a29ed91f1ced25011e07985,4b5d91666a29ed91f1ced25011e07985
seller_id,634964b17796e64304cadf1ad3050fb7,070d165398b553f3b4b851c216b8a358,070d165398b553f3b4b851c216b8a358
shipping_limit_date,2018-08-23 10:44:00,2018-06-26 19:31:00,2018-06-26 19:31:00
price,120.0,235.9,235.9
freight_value,17.03,40.73,40.73


In [48]:
# Remove order items whose order_id does not exist in orders

before = len(order_items)

order_items = order_items[
    order_items["order_id"].isin(orders["order_id"])
].reset_index(drop=True)

datasets["order_items"] = order_items

after = len(order_items)

print(f"Order items before: {before:,}")
print(f"Order items after:  {after:,}")
print(f"Rows removed:       {before - after:,}")

Order items before: 112,650
Order items after:  112,647
Rows removed:       3


In [49]:
print(
    "Orphan order items remaining:",
    (~order_items["order_id"].isin(orders["order_id"])).sum()
)

Orphan order items remaining: 0


In [50]:
# Investigate order items with missing seller references

orphan_seller_items = order_items[
    ~order_items["seller_id"].isin(sellers["seller_id"])
]

print(f"Orphan seller references: {len(orphan_seller_items)}")

orphan_seller_items.T

Orphan seller references: 6


,7256,10884,29027,43541,65310,105739
order_id,106ff3ba3e84e22713bf2a10c582fd94,18ca340d8c5cef7fba3c9c18efce43e9,4209568bdd63e8210d5697d19daa273a,62f8b4d7c208d43df56f66ad44f2a4e4,95469d3451e471f3e68f18903e341d16,f0480d88b450df6f626639e82382b722
order_item_id,1,1,1,1,1,1
product_id,a04087ab6a96ffa041f8a2701a72b616,3bbb1f94c6871212f10e8c25012a8e19,10097e23f7a5bf3ce7ec60cfa8603f4d,f2b264b1a4602b87d64eebefc899c1d0,a62e25e09e05e6faf31d90c6ec1aa3d1,a09fb9f597a4b8a13ab0c72d70c77081
seller_id,"""53243585a1d6dc2643021fd1853d8905""","""2138ccb85b11a4ec1e37afbd1c8eda1f""","""5d0363b33554b373851fc1622e4d5f3c""","""1b8356dabde1d35e17cef975c3f82730""","""634964b17796e64304cadf1ad3050fb7""","""6560211a19b47992c3666cc44a7e94c0"""
shipping_limit_date,2018-04-12 01:35:00,2017-02-21 13:18:00,2018-06-21 18:57:00,2017-04-07 14:50:00,2018-01-30 12:19:00,2017-04-06 21:42:00
price,790.0,18.9,92.0,79.99,108.0,78.0
freight_value,37.32,14.52,24.25,11.95,17.33,14.72


In [51]:
# Check quoted seller IDs

for seller_id in orphan_seller_items["seller_id"].unique():
    cleaned_id = seller_id.strip('"')

    print("Original :", repr(seller_id))
    print("Cleaned  :", repr(cleaned_id))
    print("Found in sellers:", cleaned_id in sellers["seller_id"].values)
    print()

Original : '"53243585a1d6dc2643021fd1853d8905"'
Cleaned  : '53243585a1d6dc2643021fd1853d8905'
Found in sellers: True

Original : '"2138ccb85b11a4ec1e37afbd1c8eda1f"'
Cleaned  : '2138ccb85b11a4ec1e37afbd1c8eda1f'
Found in sellers: True

Original : '"5d0363b33554b373851fc1622e4d5f3c"'
Cleaned  : '5d0363b33554b373851fc1622e4d5f3c'
Found in sellers: True

Original : '"1b8356dabde1d35e17cef975c3f82730"'
Cleaned  : '1b8356dabde1d35e17cef975c3f82730'
Found in sellers: True

Original : '"634964b17796e64304cadf1ad3050fb7"'
Cleaned  : '634964b17796e64304cadf1ad3050fb7'
Found in sellers: True

Original : '"6560211a19b47992c3666cc44a7e94c0"'
Cleaned  : '6560211a19b47992c3666cc44a7e94c0'
Found in sellers: True



In [52]:
# Remove unnecessary quotation marks from seller IDs

sellers["seller_id"] = sellers["seller_id"].str.strip('"')
order_items["seller_id"] = order_items["seller_id"].str.strip('"')

datasets["sellers"] = sellers
datasets["order_items"] = order_items

print("Seller IDs standardized successfully.")

Seller IDs standardized successfully.


In [53]:
# Re-check Order Items → Sellers relationship

orphan_seller_items = order_items[
    ~order_items["seller_id"].isin(sellers["seller_id"])
]

print(f"Order items with missing seller reference: {len(orphan_seller_items)}")

Order items with missing seller reference: 0


In [54]:
# Investigate payments with missing order references

orphan_payments = order_payments[
    ~order_payments["order_id"].isin(orders["order_id"])
]

print(f"Orphan payment records: {len(orphan_payments)}")

orphan_payments.T

Orphan payment records: 5


,63478,71580,79970,86762,96441
order_id,6d34c087b843ae0fdb296f15d948e45f,6d34c087b843ae0fdb296f15d948e45f,1a9b5dd397fc3ec9e54ff4b9a738a37e,6d34c087b843ae0fdb296f15d948e45f,6d34c087b843ae0fdb296f15d948e45f
payment_sequential,4,2,1,1,3
payment_type,voucher,voucher,credit_card,voucher,voucher
payment_installments,1,1,3,1,1
payment_value,3.26,300.0,137.03,50.0,200.0


In [55]:
# Remove payments belonging to non-existent orders

before = len(order_payments)

order_payments = order_payments[
    order_payments["order_id"].isin(orders["order_id"])
].reset_index(drop=True)

datasets["order_payments"] = order_payments

after = len(order_payments)

print(f"Payments before: {before:,}")
print(f"Payments after:  {after:,}")
print(f"Rows removed:   {before - after:,}")

Payments before: 103,886
Payments after:  103,881
Rows removed:   5


In [56]:
print(
    "Orphan payments remaining:",
    (~order_payments["order_id"].isin(orders["order_id"])).sum()
)

Orphan payments remaining: 0


In [57]:
# Investigate reviews with missing order references

orphan_reviews = order_reviews[
    ~order_reviews["order_id"].isin(orders["order_id"])
]

print(f"Orphan review records: {len(orphan_reviews)}")

orphan_reviews.T

Orphan review records: 8


,3428,9416,24056,51648,52129,65444,68908,72339
review_id,18d76645b2372fec665a35c9c57bb6f7,3014ed4f2650add0fe18e3e5e44479d5,0f514e75e16e937fda3c5076f28b0e34,8391566dcea7ecc5bf0b7cbd70d9b767,2a2b53d78d4348062fd52573eec0f787,3b798fcb93d50e5011b047359a1b81a3,124fdac4d5f25962952cd619a09be2d1,0936a1f2180f8b9f9b84147f09e97df9
order_id,"""1cff0fb86f1f68d69f65f7a9746f2096""",6d34c087b843ae0fdb296f15d948e45f,"""37f6cceb272b1ae10ed4bd8cc0d4a98a""","""d3f5bfffaeea59a28e11e01a94a25fdf""",1a9b5dd397fc3ec9e54ff4b9a738a37e,"""afd09879ff7489d300d0ebeab6348e35""","""d2ebecd13598667c0e664cf7c3eace01""","""3f606b2bc4208fc4186a5105ccef5197"""
review_score,2,5,5,5,4,1,1,5
review_comment_title,NaN,NaN,NaN,recomendo,NaN,NaN,NaN,NaN
review_comment_message,Produto não se parece com a foto.,NaN,NaN,produto entregue tudo cetinho.\r\nparabens loj...,NaN,Não entregou nem entra contato,NaN,Entrega antes do combinado. Sem problemas.
review_creation_date,2018-01-06 00:00:00,2018-06-30 00:00:00,2018-08-11 00:00:00,2018-05-10 00:00:00,2018-08-24 00:00:00,2017-11-18 00:00:00,2018-02-25 00:00:00,2017-10-24 00:00:00
review_answer_timestamp,2018-01-08 20:24:00,2018-07-02 11:53:00,2018-08-14 14:09:00,2018-05-22 23:03:00,2018-08-27 12:51:00,2017-11-20 07:19:00,2018-02-25 21:58:00,2017-10-24 15:46:00


In [58]:
# Check whether quoted review order IDs exist without quotes

orphan_review_ids = orphan_reviews["order_id"].dropna().unique()

for order_id in orphan_review_ids:
    cleaned_id = order_id.strip('"')

    print("Original :", repr(order_id))
    print("Cleaned  :", repr(cleaned_id))
    print("Found in orders:", cleaned_id in orders["order_id"].values)
    print()

Original : '"1cff0fb86f1f68d69f65f7a9746f2096"'
Cleaned  : '1cff0fb86f1f68d69f65f7a9746f2096'
Found in orders: True

Original : '6d34c087b843ae0fdb296f15d948e45f'
Cleaned  : '6d34c087b843ae0fdb296f15d948e45f'
Found in orders: False

Original : '"37f6cceb272b1ae10ed4bd8cc0d4a98a"'
Cleaned  : '37f6cceb272b1ae10ed4bd8cc0d4a98a'
Found in orders: True

Original : '"d3f5bfffaeea59a28e11e01a94a25fdf"'
Cleaned  : 'd3f5bfffaeea59a28e11e01a94a25fdf'
Found in orders: True

Original : '1a9b5dd397fc3ec9e54ff4b9a738a37e'
Cleaned  : '1a9b5dd397fc3ec9e54ff4b9a738a37e'
Found in orders: False

Original : '"afd09879ff7489d300d0ebeab6348e35"'
Cleaned  : 'afd09879ff7489d300d0ebeab6348e35'
Found in orders: True

Original : '"d2ebecd13598667c0e664cf7c3eace01"'
Cleaned  : 'd2ebecd13598667c0e664cf7c3eace01'
Found in orders: True

Original : '"3f606b2bc4208fc4186a5105ccef5197"'
Cleaned  : '3f606b2bc4208fc4186a5105ccef5197'
Found in orders: True



In [59]:
# Standardize review order IDs

order_reviews["order_id"] = order_reviews["order_id"].str.strip('"')

datasets["order_reviews"] = order_reviews

print("Review order IDs standardized successfully.")

Review order IDs standardized successfully.


In [60]:
# Remove reviews belonging to non-existent orders

before = len(order_reviews)

order_reviews = order_reviews[
    order_reviews["order_id"].isin(orders["order_id"])
].reset_index(drop=True)

datasets["order_reviews"] = order_reviews

after = len(order_reviews)

print(f"Reviews before: {before:,}")
print(f"Reviews after:  {after:,}")
print(f"Rows removed:   {before - after:,}")

Reviews before: 99,224
Reviews after:  99,222
Rows removed:   2


In [61]:
print(
    "Orphan reviews remaining:",
    (~order_reviews["order_id"].isin(orders["order_id"])).sum()
)

Orphan reviews remaining: 0


In [62]:
# Final row counts after cleaning

print("FINAL DATASET ROW COUNTS")
print("=" * 40)

for name, df in datasets.items():
    print(f"{name:22} {len(df):,}")

FINAL DATASET ROW COUNTS
customers              99,441
geolocation            738,332
order_items            112,647
order_payments         103,881
order_reviews          99,222
orders                 99,441
products               32,951
sellers                3,095
category_translation   71


In [63]:
# Final duplicate check

print("FINAL DUPLICATE CHECK")
print("=" * 40)

for name, df in datasets.items():
    print(f"{name:22} {df.duplicated().sum():,}")

FINAL DUPLICATE CHECK
customers              0
geolocation            0
order_items            0
order_payments         0
order_reviews          0
orders                 0
products               0
sellers                0
category_translation   0


In [65]:
# Final missing-value check

print("FINAL MISSING-VALUE CHECK")
print("=" * 50)

for name, df in datasets.items():
    missing = df.isna().sum()
    missing = missing[missing > 0]

    print(f"\n{name}")

    if len(missing) == 0:
        print("No missing values")
    else:
        print(missing)

FINAL MISSING-VALUE CHECK

customers
No missing values

geolocation
No missing values

order_items
No missing values

order_payments
No missing values

order_reviews
review_comment_title      87654
review_comment_message    58245
dtype: int64

orders
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64

products
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

sellers
No missing values

category_translation
No missing values
